# Bonus Day 2 Workbook: Gapminder Visual Analysis

This optional workbook extends Day 2 with public-data visualization. We will use the Gapminder teaching dataset to compare countries, continents, and historical changes over time.

The main focus is not only *how* to make charts, but *why* a particular chart type is useful for a particular analytical task.


## Learning Outcomes

By the end of this workbook, you should be able to:

- load and inspect the Gapminder dataset in a reproducible notebook
- explain what each column represents in plain language
- choose line charts, bar charts, box plots, scatter plots, and interactive charts for suitable questions
- create readable charts with Matplotlib, Seaborn, and Plotly
- compare trends over time without overclaiming causality
- write a short public-data visual story with caveats


## How to Use This Workbook

Run the notebook from top to bottom. Read the Markdown explanations, run each code cell, and pause at the guided practice prompts.

This dataset is historical. It ends in 2007, so it should not be used as a current source for population, GDP, or life expectancy.


## Setup

We use the same core visualization libraries from Day 2:

- Pandas for tabular data
- Matplotlib for clear static charts and direct control
- Seaborn for polished statistical comparison charts
- Plotly for interactive exploration and hover details

Plotly is usually available in Google Colab. If it is not available in another environment, the Plotly cells will print a note instead of stopping the whole notebook.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.max_open_warning"] = 50

try:
    import plotly.express as px
    import plotly.io as pio

    try:
        pio.renderers.default = "colab"
    except Exception:
        pass

    PLOTLY_AVAILABLE = True
    print("Plotly is available.")
except ImportError:
    px = None
    PLOTLY_AVAILABLE = False
    print("Plotly is not available in this environment. The Plotly sections will print a note instead.")


## Load the Gapminder Dataset

The loading cell tries local repository paths first. If the file is not present, it falls back to the raw GitHub URL so the notebook can run in Google Colab without manual upload.


In [ ]:
from pathlib import Path

GAPMINDER_URL = "https://raw.githubusercontent.com/ValRCS/RTU_Python_Data_Analysis_Visualization_May_2026/main/data/gapminder.csv"

local_paths = [
    Path("../../data/gapminder.csv"),  # when the notebook runs from notebooks/day_2/
    Path("../data/gapminder.csv"),     # when the notebook runs from notebooks/
    Path("data/gapminder.csv"),        # when the notebook runs from the repository root
]

for local_path in local_paths:
    if local_path.exists():
        gapminder = pd.read_csv(local_path)
        print(f"Loaded local file: {local_path}")
        break
else:
    gapminder = pd.read_csv(GAPMINDER_URL)
    print("Loaded remote file from GitHub.")

gapminder.head()


## What Is the Gapminder Dataset?

Each row describes one country in one year. The dataset repeats countries across years, so it is useful for time trends.

Important columns:

| Column | Plain-language meaning | Example chart use |
| --- | --- | --- |
| `country` | Country name | Filter or label lines and points |
| `continent` | Broad region | Compare groups by color |
| `year` | Observation year | Put on the x-axis for trends |
| `lifeExp` | Life expectancy at birth | Track or compare wellbeing over time |
| `pop` | Population | Size bubbles or summarize total population |
| `gdpPercap` | GDP per person | Compare economic level with life expectancy |


In [ ]:
column_notes = pd.DataFrame({
    "column": ["country", "continent", "year", "lifeExp", "pop", "gdpPercap"],
    "meaning": [
        "Country name",
        "Continent or broad region",
        "Observation year",
        "Life expectancy at birth",
        "Population",
        "GDP per person",
    ],
    "chart_role": [
        "Label or filter",
        "Group or color",
        "Time axis",
        "Numeric outcome",
        "Bubble size or total",
        "Numeric comparison",
    ],
})

column_notes


## Subchapter 1: Explore the Data Before Charting

### Goal

Before choosing charts, we need to understand the rows, columns, data types, missing values, and time coverage. This is similar to checking a spreadsheet before building charts from it.


### Instructor Demonstration: First Rows and Dataset Size

Start with the first few rows and the number of rows and columns. This tells us what one observation looks like and how much data we have.


In [ ]:
gapminder.head()


In [ ]:
row_count, column_count = gapminder.shape
print(f"The dataset has {row_count:,} rows and {column_count} columns.")


### Instructor Demonstration: Data Types

Data types matter because chart choices depend on whether a column is categorical, numeric, or time-like.


In [ ]:
gapminder.info()


### Instructor Demonstration: Numeric Summary

`describe()` gives a first view of numeric ranges. Notice that `year`, `lifeExp`, `pop`, and `gdpPercap` are numeric, but they play different roles in charts.


In [ ]:
gapminder.describe()


### Instructor Demonstration: Missing Values

Missing values can affect charts. If values are missing, some rows may disappear from a chart or summary.


In [ ]:
gapminder.isna().sum()


### Instructor Demonstration: Time Coverage and Groups

This dataset has repeated measurements every 5 years. We should check which years and continents are present before charting trends.


In [ ]:
years = [int(year) for year in sorted(gapminder["year"].unique())]

print(f"First year: {min(years)}")
print(f"Last year: {max(years)}")
print(f"Number of time points: {len(years)}")
print(years)


In [ ]:
gapminder["continent"].value_counts().sort_index()


In [ ]:
country_count = gapminder["country"].nunique()
print(f"Number of countries in the dataset: {country_count}")


### Guided Practice

Run the next cell, then change the country name to another country in the dataset. If your first choice is not present, choose a country from the output of `gapminder["country"].unique()`.


In [ ]:
selected_country = "Japan"

gapminder[gapminder["country"] == selected_country].head()


### Check Your Understanding

- What does one row represent?
- Which columns are categories?
- Which columns are numeric measurements?
- Why is the phrase "historical teaching dataset ending in 2007" important?


### Independent Mini Task

Display all rows for one country that interests you. Write one sentence below the table explaining what the rows represent.


In [ ]:
my_country = "Brazil"

gapminder[gapminder["country"] == my_country]


### Common Mistake

A common mistake is to treat each row as a different country. In this dataset, countries repeat across years, so the row count is larger than the number of countries.


### Interpretation

Gapminder is especially useful for visualization because it has:

- time (`year`)
- groups (`country`, `continent`)
- outcomes (`lifeExp`, `gdpPercap`, `pop`)

That combination lets us ask trend, comparison, ranking, and relationship questions.


## Subchapter 2: Match the Chart Type to the Question

### Goal

Good visualization starts with a question. The chart type should match the structure of the data and the task.


### Chart Choice Guide

| Analytical task | Useful chart | Why it fits |
| --- | --- | --- |
| Show change over time for one country | Line chart | Time is ordered, and the line emphasizes direction |
| Compare trends for several groups | Multi-line chart | Each line shows one group over the same time axis |
| Compare one value across categories | Bar chart | Lengths make category comparison easy |
| Compare spread across groups | Box plot | Shows median, spread, and possible outliers |
| Compare two numeric variables | Scatter plot | Each point shows a pair of values |
| Explore details by hovering | Plotly interactive chart | Hover text reveals details without cluttering labels |

Avoid starting with "I want a fancy chart." Start with "What do I need the chart to help me understand?"


### Instructor Demonstration: A Line Chart for One Country

Question: *How did life expectancy change over time in Japan?*

A line chart is justified because `year` is ordered and `lifeExp` is a numeric value measured repeatedly over time.


In [ ]:
japan = gapminder[gapminder["country"] == "Japan"]

japan[["year", "lifeExp", "gdpPercap", "pop"]]


In [ ]:
plt.figure(figsize=(8, 4.5))

plt.plot(japan["year"], japan["lifeExp"], marker="o")

plt.title("Japan's Life Expectancy Rose Strongly in This Historical Dataset")
plt.xlabel("Year")
plt.ylabel("Life expectancy")
plt.ylim(25, 90)
plt.show()


### Instructor Demonstration: A Line Chart Is Not Just Decoration

The line helps us see direction and pace. The markers remind us that the dataset has observations every 5 years, not every single year.


In [ ]:
life_start = japan.loc[japan["year"] == japan["year"].min(), "lifeExp"].iloc[0]
life_end = japan.loc[japan["year"] == japan["year"].max(), "lifeExp"].iloc[0]
life_change = life_end - life_start

print(f"Japan life expectancy in {japan['year'].min()}: {life_start:.1f}")
print(f"Japan life expectancy in {japan['year'].max()}: {life_end:.1f}")
print(f"Change across the dataset: {life_change:.1f} years")


### Guided Practice

Change `country_name` and rerun the next two cells. Keep the same chart type because the task is still "show one country's change over time."


In [ ]:
country_name = "Rwanda"

country_data = gapminder[gapminder["country"] == country_name]
country_data[["year", "lifeExp", "gdpPercap", "pop"]]


In [ ]:
plt.figure(figsize=(8, 4.5))

plt.plot(country_data["year"], country_data["lifeExp"], marker="o", color="darkgreen")

plt.title(f"{country_name}: Life Expectancy Over Time")
plt.xlabel("Year")
plt.ylabel("Life expectancy")
plt.ylim(25, 90)
plt.show()


### Check Your Understanding

- Why is a line chart appropriate for life expectancy over time?
- What does each marker on the line represent?
- Why would a pie chart be a poor choice for this question?


### Independent Mini Task

Choose one country and make a life expectancy line chart. Then write one sentence describing whether the pattern is steady, uneven, or surprising.


In [ ]:
my_line_country = "India"
my_line_data = gapminder[gapminder["country"] == my_line_country]

plt.figure(figsize=(8, 4.5))
plt.plot(my_line_data["year"], my_line_data["lifeExp"], marker="o", color="purple")
plt.title(f"{my_line_country}: Life Expectancy Over Time")
plt.xlabel("Year")
plt.ylabel("Life expectancy")
plt.ylim(25, 90)
plt.show()


### Common Mistake

Do not use a line chart for unordered categories such as country names. Lines imply order and continuity. Use line charts when the x-axis has a meaningful order, such as time.


### Interpretation

Line charts are excellent for *trend questions*. They are less useful when the main question is a one-year ranking or a distribution across many countries.


## Subchapter 3: Use Seaborn for Group Trends and Comparisons

### Goal

Seaborn is helpful when we want grouped charts with readable syntax. We will summarize by continent and year before plotting, similar to creating a Pivot Table in Excel.


### Instructor Demonstration: Create a Continent-Year Summary

Question: *How did typical life expectancy change by continent?*

We use the median because it describes a typical country and is less pulled by very high or very low values than the mean.


In [ ]:
continent_year_summary = (
    gapminder
    .groupby(["continent", "year"], as_index=False)
    .agg(
        median_life_expectancy=("lifeExp", "median"),
        median_gdp_per_person=("gdpPercap", "median"),
        total_population=("pop", "sum"),
        country_count=("country", "nunique"),
    )
)

continent_year_summary.head()


### Instructor Demonstration: Multi-Line Chart with Seaborn

A multi-line chart is justified because we are comparing several time trends on the same time axis.


In [ ]:
plt.figure(figsize=(10, 5.5))

sns.lineplot(
    data=continent_year_summary,
    x="year",
    y="median_life_expectancy",
    hue="continent",
    marker="o",
)

plt.title("Median Life Expectancy Increased Across All Continents")
plt.xlabel("Year")
plt.ylabel("Median life expectancy")
plt.ylim(25, 85)
plt.legend(title="Continent")
plt.show()


### Instructor Demonstration: Compare Continents in One Year

Question: *How did continents compare in 2007?*

For a single year, a line chart is no longer the best choice. A summary table and a bar chart are more direct.


In [ ]:
gapminder_2007 = gapminder[gapminder["year"] == 2007].copy()
gapminder_2007["pop_millions"] = gapminder_2007["pop"] / 1_000_000

continent_2007 = (
    gapminder_2007
    .groupby("continent", as_index=False)
    .agg(
        countries=("country", "nunique"),
        median_life_expectancy=("lifeExp", "median"),
        median_gdp_per_person=("gdpPercap", "median"),
        total_population=("pop", "sum"),
    )
    .sort_values("median_life_expectancy", ascending=False)
)

continent_2007


In [ ]:
plt.figure(figsize=(8, 4.5))

sns.barplot(
    data=continent_2007,
    x="median_life_expectancy",
    y="continent",
    color="steelblue",
)

plt.title("In 2007, Median Life Expectancy Varied by Continent")
plt.xlabel("Median life expectancy")
plt.ylabel("Continent")
plt.xlim(0, 90)
plt.show()


### Instructor Demonstration: Box Plot for Spread

Question: *Were all countries within a continent similar in 2007?*

A bar chart shows one summary value. A box plot is better when we want to see spread and differences among countries inside each continent.


In [ ]:
plt.figure(figsize=(9, 5))

sns.boxplot(
    data=gapminder_2007,
    x="continent",
    y="lifeExp",
)

plt.title("A Box Plot Shows Within-Continent Spread in 2007")
plt.xlabel("Continent")
plt.ylabel("Life expectancy")
plt.ylim(35, 85)
plt.show()


### Guided Practice

Change the summary from median life expectancy to median GDP per person. A bar chart is still appropriate because we are comparing one summary value across categories.


In [ ]:
continent_2007_gdp = continent_2007.sort_values("median_gdp_per_person", ascending=False)

plt.figure(figsize=(8, 4.5))
sns.barplot(
    data=continent_2007_gdp,
    x="median_gdp_per_person",
    y="continent",
    color="seagreen",
)

plt.title("Median GDP per Person Also Varied by Continent in 2007")
plt.xlabel("Median GDP per person")
plt.ylabel("Continent")
plt.show()


### Check Your Understanding

- Why did we summarize before making the continent line chart?
- What does a box plot show that a bar chart hides?
- Why should we be careful when comparing continents with different numbers of countries?


### Independent Mini Task

Create a box plot of `gdpPercap` by continent for 2007. Add a clear title and axis labels. Then write one observation and one caution.


In [ ]:
plt.figure(figsize=(9, 5))

sns.boxplot(
    data=gapminder_2007,
    x="continent",
    y="gdpPercap",
)

plt.title("GDP per Person Has a Wide Spread Within Continents in 2007")
plt.xlabel("Continent")
plt.ylabel("GDP per person")
plt.show()


### Common Mistake

Seaborn can calculate summaries automatically in some charts, but beginners should often create and inspect the summary table first. This makes the analysis easier to explain.


### Interpretation

Seaborn is especially useful for grouped visual questions. Use it when categories, colors, and distributions matter.


## Subchapter 4: Rankings and Change with Bar Charts

### Goal

Bar charts are useful when the task is comparison across categories. In this subchapter we use them for two different tasks: ranking countries in one year and ranking historical change.


### Instructor Demonstration: Top 10 Life Expectancy Values in 2007

Question: *Which countries had the highest life expectancy in 2007 within this dataset?*

A horizontal bar chart is justified because country names are categories and the bar lengths are easy to compare.


In [ ]:
top_life_2007 = (
    gapminder_2007
    .sort_values("lifeExp", ascending=False)
    .head(10)
)

top_life_2007[["country", "continent", "lifeExp", "gdpPercap"]]


In [ ]:
plt.figure(figsize=(8, 5))

plt.barh(top_life_2007["country"], top_life_2007["lifeExp"], color="teal")
plt.gca().invert_yaxis()

plt.title("Top 10 Life Expectancy Values in 2007")
plt.xlabel("Life expectancy")
plt.ylabel("Country")
plt.xlim(75, 85)
plt.show()


### Instructor Demonstration: Historical Change from 1952 to 2007

Question: *Where did life expectancy increase the most between the first and last year in this dataset?*

This is a change question. We first calculate the difference, then use a bar chart for the ranking.


In [ ]:
start_year = gapminder["year"].min()
end_year = gapminder["year"].max()

life_change = (
    gapminder[gapminder["year"].isin([start_year, end_year])]
    .pivot(index=["country", "continent"], columns="year", values="lifeExp")
    .reset_index()
)

life_change["life_expectancy_change"] = life_change[end_year] - life_change[start_year]
life_change = life_change.sort_values("life_expectancy_change", ascending=False)

life_change.head(10)


In [ ]:
top_change = life_change.head(10)

plt.figure(figsize=(8, 5))

sns.barplot(
    data=top_change,
    x="life_expectancy_change",
    y="country",
    hue="continent",
    dodge=False,
)

plt.title("Largest Life Expectancy Increases from 1952 to 2007")
plt.xlabel("Increase in life expectancy")
plt.ylabel("Country")
plt.legend(title="Continent")
plt.show()


### Guided Practice

The next cell shows the smallest changes. This is still a bar chart task because we are comparing calculated values across country categories.


In [ ]:
smallest_change = life_change.tail(10).sort_values("life_expectancy_change")

plt.figure(figsize=(8, 5))
sns.barplot(
    data=smallest_change,
    x="life_expectancy_change",
    y="country",
    hue="continent",
    dodge=False,
)

plt.title("Smallest Life Expectancy Changes from 1952 to 2007")
plt.xlabel("Change in life expectancy")
plt.ylabel("Country")
plt.legend(title="Continent")
plt.show()


### Check Your Understanding

- Why did we calculate change before charting?
- Why is a horizontal bar chart helpful for country names?
- What important historical context might a chart like this hide?


### Independent Mini Task

Create a top 10 bar chart for GDP per person in 2007. Then write one sentence explaining why a bar chart is a reasonable choice.


In [ ]:
top_gdp_2007 = (
    gapminder_2007
    .sort_values("gdpPercap", ascending=False)
    .head(10)
)

plt.figure(figsize=(8, 5))
sns.barplot(
    data=top_gdp_2007,
    x="gdpPercap",
    y="country",
    color="darkorange",
)

plt.title("Top 10 GDP per Person Values in 2007")
plt.xlabel("GDP per person")
plt.ylabel("Country")
plt.show()


### Common Mistake

Do not use a bar chart with too many categories at once. If there are 142 countries, a full country bar chart becomes unreadable. Filter to a meaningful top 10, bottom 10, or selected group.


### Interpretation

Bar charts are strong for ranking and category comparison. They are weaker for detailed time trends because they do not naturally show continuity over time.


## Subchapter 5: Relationships and Interactive Exploration

### Goal

Scatter plots are useful when we want to compare two numeric variables. Plotly is useful when we want to explore details with hover text or animation.


### Instructor Demonstration: Static Scatter Plot

Question: *In 2007, how did GDP per person relate to life expectancy?*

A scatter plot is justified because each point has two numeric values: `gdpPercap` and `lifeExp`. Color shows continent. Bubble size can show population, but it should be used carefully because it adds visual complexity.


In [ ]:
plt.figure(figsize=(10, 5.5))

sns.scatterplot(
    data=gapminder_2007,
    x="gdpPercap",
    y="lifeExp",
    hue="continent",
    size="pop_millions",
    sizes=(30, 500),
    alpha=0.75,
)

plt.xscale("log")
plt.title("GDP per Person and Life Expectancy Are Related, But Not Perfectly")
plt.xlabel("GDP per person (log scale)")
plt.ylabel("Life expectancy")
plt.legend(title="Continent / population", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.show()


### Interpretation Note

This chart can show association, not proof that one variable causes the other. Many historical, social, policy, and measurement factors are not shown in the chart.


### Instructor Demonstration: Plotly Interactive Line Chart

Plotly is justified when hovering helps the learner or audience inspect details without adding too many labels directly onto the chart.


In [ ]:
selected_countries = ["China", "India", "United States", "Brazil", "Rwanda", "Japan"]
selected_country_data = gapminder[gapminder["country"].isin(selected_countries)]

if PLOTLY_AVAILABLE:
    fig = px.line(
        selected_country_data,
        x="year",
        y="lifeExp",
        color="country",
        markers=True,
        title="Interactive Trend: Life Expectancy for Selected Countries",
        labels={
            "year": "Year",
            "lifeExp": "Life expectancy",
            "country": "Country",
        },
    )
    fig.show()
else:
    print("Plotly is not available here. In Colab, this cell should show an interactive line chart.")


### Instructor Demonstration: Plotly Bubble Scatter

This chart combines several variables:

- x-axis: GDP per person
- y-axis: life expectancy
- color: continent
- bubble size: population
- hover text: country name and values

It is powerful, but it also asks the reader to process more information.


In [ ]:
if PLOTLY_AVAILABLE:
    fig = px.scatter(
        gapminder_2007,
        x="gdpPercap",
        y="lifeExp",
        color="continent",
        size="pop_millions",
        hover_name="country",
        log_x=True,
        size_max=55,
        title="Interactive 2007 View: GDP per Person, Life Expectancy, and Population",
        labels={
            "gdpPercap": "GDP per person",
            "lifeExp": "Life expectancy",
            "continent": "Continent",
            "pop_millions": "Population (millions)",
        },
    )
    fig.show()
else:
    print("Plotly is not available here. In Colab, this cell should show an interactive bubble chart.")


### Instructor Demonstration: Plotly Animation Over Time

Animation is justified when the main question is how a relationship changes over time. Use it for exploration, then write a clear conclusion in words.


In [ ]:
gapminder_for_plotly = gapminder.copy()
gapminder_for_plotly["pop_millions"] = gapminder_for_plotly["pop"] / 1_000_000

if PLOTLY_AVAILABLE:
    fig = px.scatter(
        gapminder_for_plotly,
        x="gdpPercap",
        y="lifeExp",
        animation_frame="year",
        animation_group="country",
        color="continent",
        size="pop_millions",
        hover_name="country",
        log_x=True,
        size_max=50,
        range_y=[20, 90],
        range_x=[gapminder_for_plotly["gdpPercap"].min(), gapminder_for_plotly["gdpPercap"].max()],
        title="Animated Gapminder View: Countries Move Across Time",
        labels={
            "gdpPercap": "GDP per person",
            "lifeExp": "Life expectancy",
            "continent": "Continent",
            "pop_millions": "Population (millions)",
            "year": "Year",
        },
    )
    fig.show()
else:
    print("Plotly is not available here. In Colab, this cell should show an animated interactive chart.")


### Guided Practice

Change the Plotly line chart to show a different set of countries. Keep the number of countries small so the chart remains readable.


In [ ]:
practice_countries = ["Germany", "Norway", "South Africa", "Mexico"]
practice_country_data = gapminder[gapminder["country"].isin(practice_countries)]

if PLOTLY_AVAILABLE:
    fig = px.line(
        practice_country_data,
        x="year",
        y="gdpPercap",
        color="country",
        markers=True,
        title="Interactive Trend: GDP per Person for Selected Countries",
        labels={
            "year": "Year",
            "gdpPercap": "GDP per person",
            "country": "Country",
        },
    )
    fig.show()
else:
    print("Plotly is not available here. In Colab, this cell should show an interactive line chart.")


### Check Your Understanding

- Why is a scatter plot appropriate for GDP per person and life expectancy?
- What does the log scale help with?
- When does interactivity help, and when can it distract?
- Why should we avoid claiming that GDP alone causes life expectancy?


### Independent Mini Task

Create one Plotly chart that helps answer your own question. Keep it simple: one trend line chart or one scatter plot is enough. Write one sentence explaining why Plotly was useful for that task.


In [ ]:
mini_countries = ["China", "India", "Japan"]
mini_data = gapminder[gapminder["country"].isin(mini_countries)]

if PLOTLY_AVAILABLE:
    fig = px.line(
        mini_data,
        x="year",
        y="lifeExp",
        color="country",
        markers=True,
        title="Mini Task Example: Life Expectancy for Three Countries",
        labels={
            "year": "Year",
            "lifeExp": "Life expectancy",
            "country": "Country",
        },
    )
    fig.show()
else:
    print("Plotly is not available here. In Colab, this cell should show an interactive line chart.")


### Common Mistakes

- Adding too many countries to one interactive line chart.
- Using bubble size without explaining what size means.
- Forgetting that hover details are not visible in a printed report.
- Reading a scatter plot as proof of causation.


### Interpretation

Use Plotly when interaction adds value: hover labels, zooming, filtering, and animation. For a printed or slide-based report, a simpler static chart may communicate the main message more clearly.


## End-of-Day Mini Report

In this final section, assemble a short public-data visual story. The example question is:

**How did life expectancy improve over time, and how was it related to GDP per person by 2007?**

The report uses three chart types for three different tasks:

1. Seaborn line chart for continent trends over time.
2. Matplotlib or Seaborn bar chart for countries with large changes.
3. Plotly or Seaborn scatter plot for the 2007 relationship between GDP per person and life expectancy.


### Report Chart 1: Continent Trends

Task: show change over time by group.

Chart choice: multi-line chart, because each continent has values across the same ordered years.


In [ ]:
plt.figure(figsize=(10, 5.5))

sns.lineplot(
    data=continent_year_summary,
    x="year",
    y="median_life_expectancy",
    hue="continent",
    marker="o",
)

plt.title("Public-Data Story: Median Life Expectancy Rose Across Continents")
plt.xlabel("Year")
plt.ylabel("Median life expectancy")
plt.ylim(25, 85)
plt.legend(title="Continent")
plt.show()


### Report Chart 2: Largest Country-Level Changes

Task: compare calculated change across countries.

Chart choice: horizontal bar chart, because it ranks country categories by one numeric value.


In [ ]:
plt.figure(figsize=(8, 5))

sns.barplot(
    data=top_change,
    x="life_expectancy_change",
    y="country",
    hue="continent",
    dodge=False,
)

plt.title("Countries with the Largest Life Expectancy Increases, 1952 to 2007")
plt.xlabel("Increase in life expectancy")
plt.ylabel("Country")
plt.legend(title="Continent")
plt.show()


### Report Chart 3: 2007 Relationship Between GDP and Life Expectancy

Task: compare two numeric variables and inspect country details.

Chart choice: interactive scatter plot, because hover text helps identify countries without labeling every point.


In [ ]:
if PLOTLY_AVAILABLE:
    fig = px.scatter(
        gapminder_2007,
        x="gdpPercap",
        y="lifeExp",
        color="continent",
        size="pop_millions",
        hover_name="country",
        log_x=True,
        size_max=55,
        title="2007: GDP per Person and Life Expectancy by Country",
        labels={
            "gdpPercap": "GDP per person",
            "lifeExp": "Life expectancy",
            "continent": "Continent",
            "pop_millions": "Population (millions)",
        },
    )
    fig.show()
else:
    plt.figure(figsize=(10, 5.5))
    sns.scatterplot(
        data=gapminder_2007,
        x="gdpPercap",
        y="lifeExp",
        hue="continent",
        alpha=0.75,
    )
    plt.xscale("log")
    plt.title("2007: GDP per Person and Life Expectancy by Country")
    plt.xlabel("GDP per person (log scale)")
    plt.ylabel("Life expectancy")
    plt.legend(title="Continent", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.show()


### Example Findings

Based on the charts above, a careful mini report might say:

- Median life expectancy increased across all continents between 1952 and 2007 in this dataset.
- The pace and level of improvement differed by continent.
- Some countries had very large increases in life expectancy, but a bar chart alone does not explain why.
- In 2007, countries with higher GDP per person often had higher life expectancy, but the relationship was not perfect.
- These charts describe historical data ending in 2007 and should not be treated as current indicators.


### Your Mini Report Task

Create your own short report with:

- one analytical question
- two or three charts
- one sentence under each chart explaining why that chart type was suitable
- three to five findings
- one caution about the data or interpretation


## Common Mistakes and Troubleshooting

- **Choosing a chart before choosing a question:** Start with the task, then select the chart.
- **Using line charts for unordered categories:** Lines imply a meaningful order.
- **Showing too many countries at once:** Filter to selected countries or a top 10 list.
- **Forgetting the historical caveat:** This dataset ends in 2007.
- **Confusing association with causation:** A scatter plot does not prove why a pattern exists.
- **Leaving labels unclear:** Always include a title and axis labels.
- **Making interactive charts too busy:** Hover text is useful, but too many colors, sizes, and facets can overwhelm the reader.


## Reflection

Answer these prompts in Markdown:

- Which chart type felt most useful for time trends?
- Which chart type was best for ranking countries?
- Which chart type helped compare spread within continents?
- When did Plotly interactivity add value?
- What is one conclusion you can make from the data?
- What is one conclusion you should avoid making from the data alone?


## Optional Bonus Tasks

If you have extra time:

- Compare a country from Europe with a country from Asia using a line chart.
- Make a box plot of GDP per person by continent for a different year.
- Create a Plotly animation and write one sentence about what changes over time.
- Make a top 10 chart for population in 2007.
- Save one Matplotlib chart as a PNG using `plt.savefig("gapminder_chart.png", dpi=150)` before `plt.show()`.


## Run All Checklist

Before calling the workbook complete:

- The notebook runs from top to bottom.
- Data loads locally or from the raw GitHub URL.
- The dataset caveat ending in 2007 is stated.
- Matplotlib, Seaborn, and Plotly are each used for a justified purpose.
- Charts have titles and axis labels.
- The final mini report includes chart choices, findings, and cautions.
